# 04 — UV–Vis and Transition Assignments

Parse synthetic TD-DFT states, construct a broadened spectrum in energy, and prepare a conservative assignment table.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
print("Repository root:", ROOT)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from orca_reviewer.parsers import parse_tddft_states
from orca_reviewer.spectra import gaussian_broaden, energy_to_wavelength

states = pd.DataFrame(parse_tddft_states(ROOT / "sample_data" / "sample_tddft.out"))
states

In [ ]:
energies = states["energy_ev"].to_numpy()
strengths = states["oscillator_strength"].fillna(0).to_numpy()
grid_ev = np.linspace(1.5, 5.5, 2500)
y = gaussian_broaden(grid_ev, energies, strengths, fwhm=0.25)

plt.figure(figsize=(9, 4))
plt.plot(grid_ev, y)
plt.vlines(energies, 0, strengths, alpha=0.5)
plt.xlabel("Excitation energy / eV")
plt.ylabel("Relative absorption")
plt.title("Synthetic TD-DFT absorption spectrum")
plt.tight_layout()
plt.show()

In [ ]:
assignment = states[["state", "energy_ev", "wavelength_nm", "oscillator_strength", "transitions"]].copy()
assignment["NTO_hole"] = "inspect"
assignment["NTO_particle"] = "inspect"
assignment["assignment"] = "unassigned"
assignment["confidence"] = "low until orbitals/NTOs are inspected"
assignment

## Interpretation rule

Do not label a transition from the largest orbital pair alone. Inspect the full state composition and NTO hole/particle localization, especially for open-shell metal complexes.